# 요금제 EDA — 추천 서비스 관점

추천 엔진에 들어갈 컬럼을 하나씩 확인하는 것이 목적이다. 모든 섹션을 같은 순서로 정리했다.

> **① 무엇을 확인했나 → ② 데이터가 어떻게 나왔나 → ③ 파이프라인에 무엇을 반영할 것인가**

분포 그림만 모아두면 나중에 추천 결과가 이상할 때 근거를 찾기 어렵다.
이렇게 정리해두면 하드필터와 비용 계산식마다 그렇게 짠 이유가 셀로 남는다.

### 목차

| # | 섹션 | 핵심 질문 |
|---|---|---|
| 1 | 불러오기·구조 확인 | 한 줄이 요금제 1건을 뜻하는 게 맞나? |
| 2 | 빈 값 확인 | 빈 칸이 "모름"인가 "없음"인가 "무제한"인가? |
| 3 | 조건 필터 점검 | 조건을 걸 때 무제한 요금제가 빠지고 있지 않나? |
| 4 | 요금 구조 | 보이는 요금으로 줄 세우면 안 되는 이유가 숫자로 나오나? |
| 4-B | 튀는 값 점검 | 조건이 지나치게 좋은 요금제가 1위에 올라와 있지 않나? |
| 5 | 추천 후보 구성 | 실제로 추천에 쓸 수 있는 요금제는 몇 개인가? |
| 6 | 비교 항목 정리 | 데이터·음성·속도·테더링을 어떤 숫자로 바꿔야 비교가 되나? |
| 7 | 가격 대비 성능 | 가성비 한계선은 어디인가? |
| 8 | 대안 있는 요금제 걸러내기 | 후보를 어디까지 줄일 수 있는가? |
| 9 | 혜택·OTT 금액 환산 | 사은품·OTT를 돈으로 환산하면 순위가 바뀌나? |
| 10 | 가입자 수 분포 | 가입자 수를 가중치 근거로 쓸 수 있나? |
| 11 | 시간에 따른 변화 | 추천한 요금제가 다음 달에도 남아 있나? |
| 12 | 가상 사용자로 확인 | 앞의 결정이 실제 추천 결과를 바꾸는가? |
| 13 | 채널 재분류 · 요금제 그룹 | 너겟·요고·다이렉트는 어느 쪽인가? 이름만 다른 같은 요금제는? |
| 14 | 결론 체크리스트 | 코드로 옮길 항목 목록 |

실행: `notebooks/` 또는 프로젝트 루트 어디서 열어도 데이터 경로를 자동으로 찾는다.

## 0. 준비

폰트·색·경로만 잡는다. 색은 **Okabe–Ito** 팔레트(색각 이상에서도 구분되는 8색)를 고정 순서로 쓴다.
차트는 "한 그림에 한 메시지" 원칙으로, 이중 축은 쓰지 않는다.

In [ ]:
import warnings, glob, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

warnings.filterwarnings("ignore")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# 한글 폰트 자동 선택 (Windows / mac / Linux)
_installed = {f.name for f in font_manager.fontManager.ttflist}
for _name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans KR", "Noto Sans CJK KR", "Noto Sans CJK JP", "DejaVu Sans"]:
    if _name in _installed:
        matplotlib.rcParams["font.family"] = _name
        break
matplotlib.rcParams["axes.unicode_minus"] = False
matplotlib.rcParams.update({
    "figure.figsize": (9, 4.0), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.titlelocation": "left", "axes.titlesize": 12, "font.size": 10,
})

# 카테고리 색: 고정 순서로만 쓰고 절대 순환시키지 않는다
C = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73", "red": "#D55E00",
     "pink": "#CC79A7", "sky": "#56B4E9", "grey": "#9AA0A6", "ink": "#3C4043"}

def find_root() -> Path:
    here = Path.cwd()
    for p in [here, *here.parents][:5]:
        if (p / "data" / "통신요금제_통합데이터_최종.csv").exists():
            return p
    raise FileNotFoundError("data/통신요금제_통합데이터_최종.csv 를 찾지 못했습니다.")

ROOT = find_root()

SAVE_FIG = False          # True 로 바꾸면 그림을 notebooks/eda_figures/ 에 PNG로도 저장한다
FIGDIR = ROOT / "notebooks" / "eda_figures"

_fig_no = [0]
def show(name: str = "fig"):
    _fig_no[0] += 1
    plt.tight_layout()
    if SAVE_FIG:
        FIGDIR.mkdir(parents=True, exist_ok=True)
        plt.savefig(FIGDIR / f"{_fig_no[0]:02d}_{name}.png", dpi=150,
                    bbox_inches="tight", facecolor="white")
    plt.show()

print("ROOT =", ROOT)

## 1. 데이터 불러오기와 구조 확인

먼저 확인할 것은 분포가 아니라 **"이 표의 한 행이 무엇인가"** 다.
여기서 어긋나면 이후 통계가 모두 어긋난다. 세 가지를 확인한다:

1. **키가 유일한가** — `plan_id` 중복이 있으면 추천 결과가 같은 요금제를 두 번 보여준다.
2. **행의 의미가 섞여 있지 않은가** — MNO 공식 요금제와 모요(MVNO) 크롤링이 한 표에 있다.
3. **이름이 같은 다른 행이 있는가** — 프로모션 변형("Npay 5000P" 류)은 사람 눈에 중복처럼 보인다.

In [ ]:
plans = pd.read_csv(ROOT / "data/통신요금제_통합데이터_최종.csv",
                    dtype={"plan_id": str, "base_plan_id": str}, encoding="utf-8-sig")
benefits = pd.read_csv(ROOT / "data/통신요금제_혜택상세_최종.csv",
                       dtype={"plan_id": str}, encoding="utf-8-sig")
ott_prices = pd.read_csv(ROOT / "data/ott_prices.csv", encoding="utf-8-sig")

print(f"요금제 {plans.shape[0]:,}행 × {plans.shape[1]}열")
print(f"혜택   {benefits.shape[0]:,}행 × {benefits.shape[1]}열")
print(f"OTT가격 {ott_prices.shape[0]}행")

print("\n[키 검사]")
print("plan_id 중복            :", plans["plan_id"].duplicated().sum())
print("plan_id 결측            :", plans["plan_id"].isna().sum())
print("(이름+요금+데이터) 중복 :", plans.duplicated(["plan_name", "monthly_fee", "data_gb"]).sum())
print("base_plan_id 당 행 수 최대:", plans.groupby("base_plan_id").size().max(), " ← 옵션 변형")
print("crawled_at 스냅샷 일자  :", sorted(plans["crawled_at"].str[:10].unique()))

In [ ]:
def audit(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "결측률%": (df.isna().mean() * 100).round(1),
        "고유값": df.nunique(dropna=True),
        "예시": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })

audit(plans)

**읽는 법.** `plan_id` 중복 0, 스냅샷 날짜 1종 → 이 파일은 **특정 시점의 요금제 카탈로그 1장**이다.
시계열이 아니다(시간축은 11장의 `changes_*.csv`가 담당).
반면 `(이름+요금+데이터)` 기준으로는 300건 남짓 겹치는데, 이건 버그가 아니라
**같은 상품의 유통 채널·프로모션 변형**이다. 추천 결과 화면에서 "똑같아 보이는 카드 3장"이
뜨는 원인이므로, 지우지 말고 **표시 단계에서 묶어야(grouping) 한다**.

> **→ 파이프라인 반영 #1**: 후보 중복 제거는 `plan_id`가 아니라
> `(plan_name, monthly_fee, data_gb)` 기준 **대표 1건 + "유사 상품 N건" 접기**로.

## 2. 빈 값 확인 — 빈 칸의 의미는 항목마다 다르다

결측률을 "채워야 할 값"으로만 보면 안 된다. 이 데이터에서 빈 칸은 최소 네 가지 의미를 가진다.

| 빈 칸의 의미 | 예시 컬럼 | 올바른 처리 |
|---|---|---|
| **무제한이라 숫자가 없음** | `data_gb`, `voice_minutes`, `sms_count` | 플래그 컬럼을 먼저 보고 ∞로 취급 |
| **해당 없음(혜택이 없음)** | `ott_options`, `gift_benefit`, `age_condition` | 0 / False로 채움 |
| **구조적으로 다른 뜻** | `discount_period_months` | MVNO=할인 없음, MNO=약정형 |
| **진짜 모름** | `data_throttle_speed`, `tethering_gb` | 결측 자체를 값으로(=`undisclosed`) |

평균 대치(mean imputation)를 적용하면 이 네 가지가 구분되지 않는다.

In [ ]:
miss = (plans.isna().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(miss.index[::-1], miss.values[::-1], color=C["blue"], height=0.68)
ax.set_xlabel("결측률 (%)"); ax.set_xlim(0, 100)
ax.set_title("항목별 빈 값 비율")
for y, v in zip(range(len(miss)), miss.values[::-1]):
    ax.text(v + 1.2, y, f"{v:.0f}", va="center", fontsize=8, color=C["ink"])
ax.grid(axis="y", visible=False)
show("missing_rate")

In [ ]:
# 결측이 '어떤 행에서 같이 비는지' — 패턴이 보이면 그건 구조이지 잡음이 아니다
cols = [c for c in plans.columns if plans[c].isna().any()]
M = plans[cols].isna().astype(np.int8).values
order = np.lexsort((M.sum(1), plans["carrier_type"].factorize()[0]))

fig, ax = plt.subplots(figsize=(10.5, 6))
ax.imshow(M[order].T, aspect="auto", cmap="Blues", interpolation="nearest", vmin=0, vmax=1)
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols, fontsize=7.5)
ax.set_xlabel("요금제 (MNO 먼저 → MVNO, 결측 개수 순 정렬)")
ax.set_title("빈 값이 어디에 몰려 있나 (진한 부분이 빈 칸)")
n_mno = (plans["carrier_type"] == "MNO").sum()
ax.axvline(n_mno, color=C["red"], lw=1.4)
ax.text(n_mno + 25, len(cols) * 0.06, "← MNO | MVNO →", color=C["red"], fontsize=9)
ax.grid(False)
show("missing_pattern")

**세로로 패턴이 갈리는 지점**이 MNO/MVNO 경계다. 결측이 무작위가 아니라 **수집 소스에 따라 갈린다**는 뜻이다.
`subscriber_count`, `mvno_brand`가 MNO 쪽에서 전부 비고, `age_condition`·`membership_grade`는 MNO 쪽에만 찬다.
이런 데이터에 공통 결측 처리(dropna, 평균 대치)를 일괄 적용하면 **한쪽 소스가 통째로 사라진다.**

In [ ]:
# 빈 칸의 뜻을 하나씩 확인한다
print("[A] 무제한이라 비어 있는 경우")
print("  data_gb 결측:", plans["data_gb"].isna().sum(),
      "| 그중 data_unlimited=True:", int(plans.loc[plans["data_gb"].isna(), "data_unlimited"].sum()))
print("  voice_minutes 결측:", plans["voice_minutes"].isna().sum(),
      "| 그중 voice_unlimited=True:", int(plans.loc[plans["voice_minutes"].isna(), "voice_unlimited"].sum()))
print("  sms_count 결측:", plans["sms_count"].isna().sum(),
      "| 그중 sms_unlimited=True:", int(plans.loc[plans["sms_count"].isna(), "sms_unlimited"].sum()))

print("\n[B] discount_period_months 결측의 뜻 — carrier_type 별로 완전히 다르다")
tbl = plans.assign(할인있음=plans["discounted_fee"] < plans["monthly_fee"],
                   기간결측=plans["discount_period_months"].isna())
print(pd.crosstab([tbl["carrier_type"], tbl["할인있음"]], tbl["기간결측"]))

**확인 결과.**

* **MVNO**: 할인가가 정상가보다 싼 행은 전부 `discount_period_months`가 채워져 있다.
  즉 MVNO에서 기간 결측 = **할인 자체가 없음**. → `fillna(0)` 이 옳다.
* **MNO**: 466건이 "선택약정 25% 할인"인데 **기간이 전부 결측**이다.
  선택약정은 관행적으로 12/24개월 약정이므로, 여기에 `fillna(0)`을 쓰면
  **MNO 실효 요금을 체계적으로 과대 계산**한다 → "MNO는 무조건 비싸다"는 잘못된 결론으로 이어진다.

MNO를 **절감액 기준선(baseline)** 으로 쓰는 설계에서 기준선이 틀리면 "얼마 아낀다"는 문구가 통째로 틀린다.

> **→ 파이프라인 반영 #2**: 할인 기간 결측 규칙을 caller마다 쓰지 말고 한 함수로 고정한다.
> MVNO → 0개월, MNO+선택약정 → 24개월(가정임을 리포트에 명시).

## 3. 조건 필터 점검 — 무제한 요금제가 빠지지 않는가

추천 서비스에서 문제가 되는 것은 **에러 없이 잘못된 후보를 내놓는** 경우다.
pandas에서 `NaN >= 20` 은 예외가 아니라 `False`이고, 이 데이터에서 무제한 요금제는 숫자가 NaN이다.
즉 **"데이터 20GB 이상"을 `data_gb >= 20` 만으로 거르면 무제한 요금제가 전부 빠진다.**

아래에서 그 차이를 수치로 확인하고, 무제한을 살리는 비교 함수를 만든다.

In [ ]:
def at_least(value: pd.Series, unlimited_flag: pd.Series, threshold: float) -> pd.Series:
    # 'threshold 이상' 의 안전한 버전 — 무제한 플래그를 먼저 살린다.
    return unlimited_flag.fillna(False) | (value.fillna(-np.inf) >= threshold)

rows = []
for th in [5, 20, 50, 100, 150]:
    naive = (plans["data_gb"] >= th).sum()                                  # 흔한 실수
    safe = at_least(plans["data_gb"], plans["data_unlimited"], th).sum()     # 올바른 필터
    rows.append({"조건": f"데이터 {th}GB 이상", "무제한 미반영": naive, "무제한 반영": safe,
                 "누락된 무제한": safe - naive,
                 "누락 비율%": round((safe - naive) / max(safe, 1) * 100, 1)})
loss = pd.DataFrame(rows)
loss

In [ ]:
fig, ax = plt.subplots()
x = np.arange(len(loss))
ax.bar(x - 0.2, loss["무제한 미반영"], width=0.38, color=C["grey"], label="무제한 미반영 (>= 만 적용)")
ax.bar(x + 0.2, loss["무제한 반영"], width=0.38, color=C["blue"], label="무제한 반영")
for xi, (a, b) in enumerate(zip(loss["무제한 미반영"], loss["무제한 반영"])):
    ax.text(xi + 0.2, b + 15, f"+{b-a}", ha="center", fontsize=9, color=C["red"])
ax.set_xticks(x); ax.set_xticklabels(loss["조건"], fontsize=9)
ax.set_ylabel("통과 요금제 수"); ax.legend(frameon=False, loc="upper right")
ax.set_title("'데이터 N GB 이상' 조건에 걸리는 요금제 수 — 무제한 처리 전후")
ax.grid(axis="x", visible=False)
show("unlimited_filter_loss")

In [ ]:
# 조건을 하나씩 더할 때 후보가 어떻게 줄어드는가
step_defs = [
    ("MVNO 전체", lambda d: pd.Series(True, index=d.index)),
    ("+ 데이터 20GB 이상", lambda d: at_least(d["data_gb"], d["data_unlimited"], 20)),
    ("+ 음성 무제한", lambda d: d["voice_unlimited"].fillna(False)),
    ("+ 다 쓴 뒤 3Mbps 이상", lambda d: d["data_throttle_speed"].map(
        lambda v: 0.0 if pd.isna(v) else (float(str(v).lower().replace("mbps", "")) if "mbps" in str(v).lower()
                                          else float(str(v).lower().replace("kbps", "")) / 1000)) >= 3),
    ("+ 테더링 제공", lambda d: d["tethering_support"].isin(["quota", "within_data"])),
    ("+ 월 3만원 이하", lambda d: d["discounted_fee"] <= 30000),
]
d0 = plans[plans["carrier_type"] == "MVNO"]
mask = pd.Series(True, index=d0.index)
labels, counts = [], []
for label, fn in step_defs:
    mask = mask & fn(d0)
    labels.append(label); counts.append(int(mask.sum()))

fig, ax = plt.subplots(figsize=(9, 4.4))
bars = ax.barh(labels[::-1], counts[::-1], color=C["blue"], height=0.62)
for y, v in enumerate(counts[::-1]):
    ax.text(v + 20, y, f"{v:,}", va="center", fontsize=9.5, color=C["ink"])
ax.set_xlabel("남은 후보 수"); ax.set_title("조건을 하나씩 더할 때 남는 요금제 수")
ax.grid(axis="y", visible=False)
show("hard_filter_funnel")

for (l, c1), c2 in zip(zip(labels[1:], counts[1:]), counts[:-1]):
    print(f"{l:22s} {c2:5,} → {c1:5,}  ({(c1-c2)/max(c2,1):+.0%})")

**해석.** 후보 수가 급격히 줄어드는 조건이 어디인지 확인한다.
급감 구간 직전에서 "이 조건을 풀면 N개가 더 보입니다" 를 제안해야 한다.
순서를 바꿔 실행해 보면 어떤 조건이 진짜 병목인지도 바로 보인다(조건 간 상관 때문에 순서에 따라 달라진다).

> **→ 파이프라인 반영 #3**: 숫자 비교가 들어가는 모든 조건 필터는 `at_least()` 같은 헬퍼를 거치게 한다.
> 그리고 **후보 0건이 되는 조건 조합은 테스트로 고정**해둔다(회귀 방지).

## 4. 요금 구조 — 보이는 요금과 실제 내는 돈은 다르다

알뜰폰 표시가는 대부분 **7개월 프로모션가**다. 사용자가 실제로 내는 금액은

$$\text{총비용}(M) = \text{할인가} \times \min(P, M) + \text{정상가} \times (M - \min(P, M))$$

($P$ = 할인 개월수, $M$ = 비교 기간). 이 식이 필요한 이유는 **순위가 얼마나 바뀌는가**로 확인한다.

In [ ]:
def discount_months(df: pd.DataFrame, mno_commitment: int = 24) -> pd.Series:
    # 할인 기간 정규화. 2장에서 확인한 결측 의미를 한 곳에 고정한다.
    p = df["discount_period_months"].copy()
    has_discount = df["discounted_fee"] < df["monthly_fee"]
    mno_select = (df["carrier_type"] == "MNO") & has_discount & p.isna()
    p = p.fillna(0.0)
    p.loc[mno_select] = mno_commitment   # 선택약정 = 약정기간 내내 유지된다는 가정
    return p

def total_cost(df: pd.DataFrame, months: int, mno_commitment: int = 24) -> pd.Series:
    p = discount_months(df, mno_commitment).clip(upper=months)
    return df["discounted_fee"] * p + df["monthly_fee"] * (months - p)

for m in (6, 12, 24):
    plans[f"cost_{m}m"] = total_cost(plans, m)
    plans[f"eff_{m}m"] = plans[f"cost_{m}m"] / m      # M개월 기준 실제 월요금

mvno = plans[plans["carrier_type"] == "MVNO"]
print("알뜰폰 — 기준 기간별 실제 월요금 중앙값")
print(mvno[["discounted_fee", "eff_6m", "eff_12m", "eff_24m", "monthly_fee"]].median().round(0).to_string())

rho = mvno[["discounted_fee", "eff_24m"]].corr(method="spearman").iloc[0, 1]
print(f"\n두 기준의 순위가 얼마나 비슷한가 (1에 가까울수록 동일) = {rho:.3f}")
top50_display = set(mvno.nsmallest(50, "discounted_fee")["plan_id"])
top50_eff = set(mvno.nsmallest(50, "eff_24m")["plan_id"])
print(f"'가장 싼 50개' 겹침: {len(top50_display & top50_eff)} / 50")

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 6.0))
ax.scatter(mvno["discounted_fee"], mvno["eff_24m"], s=9, alpha=0.35,
           color=C["blue"], edgecolors="none")
lim = [0, max(mvno["discounted_fee"].max(), mvno["eff_24m"].max()) * 1.02]
ax.plot(lim, lim, color=C["grey"], lw=1.2, ls="--")
ax.text(lim[1] * 0.52, lim[1] * 0.56, "할인이 계속된다면 이 선 위에 있음", fontsize=9,
        color=C["ink"], rotation=45, rotation_mode="anchor")
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("지금 보이는 요금 (할인 적용가, 원)"); ax.set_ylabel("24개월 기준 실제 월요금 (원)")
ax.set_title("지금 보이는 요금 vs 24개월 평균 실제 요금 (대각선 위 = 나중에 오름)")
show("display_vs_effective")

In [ ]:
ratio = (mvno["monthly_fee"] / mvno["discounted_fee"].replace(0, np.nan)).dropna()
fig, ax = plt.subplots()
ax.hist(ratio.clip(upper=8), bins=40, color=C["blue"])
ax.axvline(1, color=C["grey"], lw=1.2, ls="--")
ax.axvline(ratio.median(), color=C["orange"], lw=2)
ax.text(ratio.median() + 0.12, ax.get_ylim()[1] * 0.85,
        f"중앙값 {ratio.median():.2f}배", color=C["orange"], fontsize=10)
ax.set_xlabel("할인 종료 후 요금 배수 (원래 요금 ÷ 할인 요금, 8배에서 자름)")
ax.set_ylabel("요금제 수")
ax.set_title("할인이 끝나면 요금이 몇 배가 되는가 (알뜰폰)")
show("promo_cliff_ratio")

print("할인 개월수 분포 (MVNO):")
print(mvno["discount_period_months"].value_counts().sort_index().head(10).to_string())

In [ ]:
# 프로모션 기간이 다른 요금제들의 '누적 지출' 곡선 — 언제 역전되는가
picks = (mvno[(mvno["data_gb"].between(90, 160)) & (mvno["discounted_fee"] > 0)]
         .sort_values("discounted_fee").head(4))
picks = pd.concat([picks, mvno[mvno["discount_period_months"].isna()].nsmallest(1, "monthly_fee")])
months = np.arange(1, 25)

fig, ax = plt.subplots(figsize=(9, 5))
palette = [C["blue"], C["orange"], C["green"], C["pink"], C["sky"]]
for color, (_, r) in zip(palette, picks.iterrows()):
    p = 0 if pd.isna(r["discount_period_months"]) else r["discount_period_months"]
    cum = [r["discounted_fee"] * min(p, m) + r["monthly_fee"] * (m - min(p, m)) for m in months]
    ax.plot(months, np.array(cum) / 10000, lw=2, color=color)
    ax.text(24.3, cum[-1] / 10000, f"{r['plan_name'][:16]}", fontsize=8.5, color=color, va="center")
    if 0 < p < 24:
        ax.scatter([p], [cum[int(p) - 1] / 10000], s=42, color=color, zorder=5, edgecolors="white", lw=1.2)
ax.set_xlim(1, 31); ax.set_xticks([1, 6, 7, 12, 18, 24])
ax.set_xlabel("가입 후 경과 개월"); ax.set_ylabel("누적 지출 (만원)")
ax.set_title("가입 후 실제로 낸 돈의 누적 (점 = 할인 끝나는 달)")
show("cumulative_spend")

In [ ]:
# 순위 역전: 표시가 Top15 가 실효가 기준으로 어디까지 밀리는가 (덤벨 차트)
rank_disp = mvno["discounted_fee"].rank(method="min")
rank_eff = mvno["eff_24m"].rank(method="min")
top = (mvno.assign(r_disp=rank_disp, r_eff=rank_eff)
       .nsmallest(15, "discounted_fee").sort_values("r_eff", ascending=False))

fig, ax = plt.subplots(figsize=(9.5, 6.2))
for y, (_, r) in enumerate(top.iterrows()):
    drop = r["r_eff"] - r["r_disp"]
    color = C["red"] if drop > 100 else (C["grey"] if drop > 10 else C["green"])
    ax.plot([r["r_disp"], r["r_eff"]], [y, y], color=color, lw=2, alpha=0.75, zorder=2)
    ax.scatter([r["r_disp"]], [y], s=46, color=C["ink"], zorder=3)
    ax.scatter([r["r_eff"]], [y], s=56, color=color, zorder=3)
    ax.text(r["r_eff"] * 1.12, y, f"{int(r['r_eff'])}위", va="center", fontsize=8.5, color=color)
ax.set_yticks(range(len(top)))
ax.set_yticklabels([f"{b[:6]} · {n[:20]}" for b, n in zip(top["mvno_brand"].fillna(""), top["plan_name"])],
                   fontsize=8.5)
ax.set_xscale("log"); ax.set_xlim(0.7, 2500)
ax.set_xlabel("알뜰폰 2,203개 중 순위 (로그, 왼쪽일수록 저렴)")
ax.set_title("지금 가장 싸 보이는 15개가 24개월 기준으로는 몇 위인가")
ax.grid(axis="y", visible=False)
show("rank_slope")
print("표시가 Top15 중 실효가 기준 100계단 이상 밀린 개수:",
      int(((top['r_eff'] - top['r_disp']) > 100).sum()), "/ 15")

**해석.** 위 상관값이 1.0에서 멀수록 "보이는 요금으로 줄 세운 순위"와 "실제 지출 순위"가 다르다는 뜻이다.
`가장 싼 50개`의 겹침 개수를 보면 **표시가 랭킹이 실제로 몇 자리를 잘못 놓는지**가 바로 보인다.
이게 모요/스마트초이스 대비 차별점의 **수치 근거**다 — 주장할 때 이 숫자를 그대로 쓰면 된다.

또 하나: 배수 히스토그램의 오른쪽 꼬리(정상가가 할인가의 3배 이상)는
**"1년 뒤 요금 폭탄" 후보군**이다. 리포트에 경고 문구를 띄울 대상은 이 꼬리다.

> **→ 파이프라인 반영 #4**: 순위 기준은 `eff_6m`이 아니라 **비교 기간을 사용자가 고르게 하고**
> 기본값 24개월. 보이는 요금은 화면 표시용으로만 쓰고 정렬에는 쓰지 않는다.
> 배수 ≥ 2인 요금제에는 `reason_tags`에 `promo_cliff` 플래그.

## 4-B. 튀는 값 점검 — 조건이 지나치게 좋은 요금제 확인

조건이 나쁜 행은 랭킹 아래로 내려가 눈에 띄지 않지만, 조건이 지나치게 좋은 행은 **모든 사용자의 1위**로 올라온다.
그래서 점검 대상은 후자다.

검증 방법 두 가지 — 둘 다 "외부 기준과 대조"라는 같은 아이디어다.

1. **같은 이름의 요금제가 MNO 표에도 있으면 정상가를 맞춰본다** (크롤 소스 간 교차검증)
2. **같은 데이터 구간의 정상가 중앙값과 비교한다** (내부 분포 기준)

In [ ]:
# ① 소스 간 교차검증: 같은 plan_name 이 MNO/MVNO 양쪽에 존재하는 경우
mno_ref = (plans[plans["carrier_type"] == "MNO"][["plan_name", "monthly_fee", "data_gb"]]
           .drop_duplicates("plan_name")
           .rename(columns={"monthly_fee": "MNO_정상가", "data_gb": "MNO_GB"}))
cross = (plans[plans["carrier_type"] == "MVNO"]
         [["plan_name", "mvno_brand", "monthly_fee", "discounted_fee", "data_gb", "discount_type"]]
         .merge(mno_ref, on="plan_name", how="inner"))
cross["정상가 비율"] = (cross["monthly_fee"] / cross["MNO_정상가"]).round(3)
print(f"이름이 일치하는 요금제 {len(cross)}건 중 정상가가 MNO의 50% 미만: "
      f"{(cross['정상가 비율'] < 0.5).sum()}건")
cross[cross["정상가 비율"] < 0.5]

In [ ]:
# ② 내부 분포 기준: 데이터 구간별 정상가 중앙값 대비
d = plans[plans["carrier_type"] == "MVNO"].copy()
d["구간"] = pd.cut(d["data_gb"], [0, 3, 11, 31, 71, 151, 1e9],
                 labels=["~3GB", "3~11GB", "11~31GB", "31~71GB", "71~151GB", "151GB+"])
med = d.groupby("구간")["monthly_fee"].median()
d["구간중앙값"] = d["구간"].map(med).astype(float)
d["정상가/중앙값"] = (d["monthly_fee"] / d["구간중앙값"]).round(2)

odd = d[(d["정상가/중앙값"] < 0.3) & d["구간중앙값"].notna()]
print(f"정상가가 같은 구간 중앙값의 30% 미만인 요금제: {len(odd)}건")
odd[["mvno_brand", "plan_name", "monthly_fee", "discounted_fee",
     "discount_period_months", "data_gb", "정상가/중앙값", "discount_type"]].sort_values("정상가/중앙값").head(10)

In [ ]:
# 이 값이 추천 결과에 실제로 영향을 주는지 확인
suspect = cross.loc[cross["정상가 비율"] < 0.5, "plan_name"].tolist()
print("의심 요금제:", suspect)
if suspect:
    tgt = plans[plans["plan_name"].isin(suspect) & plans["carrier_type"].eq("MVNO")]
    print(tgt[["plan_name", "monthly_fee", "discounted_fee", "discount_type",
               "discount_period_months", "data_gb", "eff_24m", "subscriber_count"]].to_string(index=False))
    rank = (plans[plans["carrier_type"] == "MVNO"]["eff_24m"].rank()
            .loc[tgt.index])
    print("\n2,203개 MVNO 중 실효요금 순위:", rank.astype(int).tolist(), "← 데이터 120GB에 이 순위면 비정상")

In [ ]:
# 이상치를 그림으로 — 같은 데이터 구간에서 혼자 아래로 떨어진 점
d_ = plans[plans["carrier_type"] == "MVNO"].dropna(subset=["data_gb"]).copy()
flag = d_["plan_name"].isin(cross.loc[cross["정상가 비율"] < 0.5, "plan_name"])
fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.scatter(d_.loc[~flag, "data_gb"], d_.loc[~flag, "monthly_fee"], s=10, alpha=0.3,
           color=C["grey"], edgecolors="none", label="정상")
ax.scatter(d_.loc[flag, "data_gb"], d_.loc[flag, "monthly_fee"], s=90, color=C["red"],
           marker="X", label="교차검증 이상 (정상가 오염 의심)", zorder=5)
for _, r in d_[flag].iterrows():
    ax.annotate(f"{r['plan_name']}  정상가 {r['monthly_fee']:,}원\n(MNO 표기 49,000원)",
                (r["data_gb"], r["monthly_fee"]), textcoords="offset points", xytext=(-18, 26),
                fontsize=9, color=C["red"], ha="right",
                arrowprops=dict(arrowstyle="->", color=C["red"], lw=1.1))
ax.set_xscale("log"); ax.set_xlabel("월 데이터 GB (log)"); ax.set_ylabel("정상 월요금 (원)")
ax.set_title("데이터 양 대비 원래 요금 — 혼자 튀는 값 확인")
ax.legend(frameon=False, loc="upper left")
show("outlier_audit")

**해석.** `너겟49`가 대표 사례다. MNO 표에는 정상가 49,000원으로 들어 있는데
모요 크롤 행에는 **정상가 자리에 페이백 후 가격 7,000원**이 들어가 있고, 할인 기간은 비어 있다.
결과적으로 **"120GB를 영구히 7,000원에"** 라는 요금제가 만들어지고,
12장 가상 사용자 검증에서 Top-1이 이 행으로 채워지고, 8장 비교에서도 수백 개를 혼자 이긴다.

한 행이 추천 결과 첫 화면을 결정한다. 그리고 이런 행은
**`discount_type`이 "페이백"인데 `discount_period_months`가 비어 있는** 패턴에서 나온다 — 규칙으로 잡을 수 있다.

> **→ 파이프라인 반영 #4-B**: 적재 단계에 **이상치 게이트** 추가.
> (a) 페이백형 할인은 정상가가 아니라 할인가 필드에만 반영, (b) 같은 이름의 MNO 정상가와 50% 이상 차이나면 격리,
> (c) 구간 중앙값 대비 30% 미만이면 `needs_review` 플래그 → **자동 삭제가 아니라 검토 큐**로 보낸다
> (73건 중엔 진짜 초저가 상품도 섞여 있다. 확인 전까지는 Top-N 노출만 보류하는 게 안전하다).
> **이 점검 없이 만든 추천 결과는 신뢰하기 어렵다.**

## 5. 추천 후보 구성

전체 2,759개 중 추천 엔진이 실제로 고를 수 있는 후보는 그보다 적다.
MNO는 **절감 기준선**이지 후보가 아니고, 연령 전용·온라인 전용은 조건부다.

In [ ]:
print(plans["carrier_type"].value_counts().to_string())
print("\n브랜드 수:", plans["mvno_brand"].nunique(), "| 망 제공사:", plans["host_mno"].unique().tolist())

pool = pd.DataFrame({
    "전체": [len(plans)],
    "MVNO(추천 후보)": [(plans["carrier_type"] == "MVNO").sum()],
    "MNO(절감 기준선)": [(plans["carrier_type"] == "MNO").sum()],
    "연령 조건 있음": [plans["age_condition"].notna().sum()],
    "온라인 전용": [plans["is_online_only"].sum()],
    "5G": [(plans["network_gen"] == "5G").sum()],
    "LTE": [(plans["network_gen"] == "LTE").sum()],
    "망세대 미상": [plans["network_gen"].isna().sum()],
}).T.rename(columns={0: "건수"})
pool["비중%"] = (pool["건수"] / len(plans) * 100).round(1)
pool

In [ ]:
brand = mvno["mvno_brand"].value_counts()
top = brand.head(15)[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top.index, top.values, color=C["blue"], height=0.68)
ax.set_xlabel("요금제 수"); ax.set_title(f"브랜드별 요금제 수 (상위 15 / 전체 {brand.size}개 브랜드)")
ax.grid(axis="y", visible=False)
share_top5 = brand.head(5).sum() / brand.sum() * 100
ax.text(0.98, 0.04, f"상위 5개 브랜드가 전체의 {share_top5:.0f}%",
        transform=ax.transAxes, ha="right", fontsize=10, color=C["ink"])
show("brand_plan_count")

In [ ]:
fig, ax = plt.subplots()
bins = np.arange(0, 135000, 5000)
ax.hist(plans.loc[plans["carrier_type"] == "MVNO", "monthly_fee"], bins=bins,
        color=C["blue"], alpha=0.85, label="MVNO")
ax.hist(plans.loc[plans["carrier_type"] == "MNO", "monthly_fee"], bins=bins,
        color=C["orange"], alpha=0.75, label="MNO")
ax.set_xlabel("정상 월요금 (원)"); ax.set_ylabel("요금제 수")
ax.legend(frameon=False)
ax.set_title("통신3사와 알뜰폰의 요금대 비교")
show("fee_distribution")

print(plans.groupby("carrier_type")["monthly_fee"].describe()[["min", "25%", "50%", "75%", "max"]].round(0).to_string())

In [ ]:
# 브랜드별 가격 분포 — '싼 브랜드'가 따로 있는가
top_brands = mvno["mvno_brand"].value_counts().head(12).index.tolist()
data = [mvno.loc[mvno["mvno_brand"] == b, "eff_24m"].dropna().values for b in top_brands]
order = np.argsort([np.median(x) for x in data])
data = [data[i] for i in order]; labels_b = [top_brands[i] for i in order]

fig, ax = plt.subplots(figsize=(9.5, 5))
bp = ax.boxplot(data, labels=labels_b, showfliers=False, patch_artist=True, widths=0.62, vert=True)
for patch in bp["boxes"]:
    patch.set_facecolor(C["blue"]); patch.set_alpha(0.6); patch.set_edgecolor(C["ink"])
for med in bp["medians"]:
    med.set_color(C["orange"]); med.set_linewidth(2)
ax.set_ylabel("24개월 기준 실제 월요금 (원)"); ax.set_title("브랜드별 요금대 (24개월 기준, 싼 순)")
plt.setp(ax.get_xticklabels(), rotation=35, ha="right", fontsize=9)
ax.grid(axis="x", visible=False)
show("brand_price_box")

In [ ]:
# 망세대 × 캐리어 구성
ct = pd.crosstab(plans["network_gen"].fillna("미상"), plans["carrier_type"])
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
ct.plot(kind="bar", stacked=True, ax=axes[0], color=[C["orange"], C["blue"]], width=0.62)
axes[0].set_title("통신망 세대별 요금제 수"); axes[0].set_xlabel(""); axes[0].set_ylabel("요금제 수")
axes[0].legend(frameon=False); axes[0].grid(axis="x", visible=False)
plt.setp(axes[0].get_xticklabels(), rotation=0)

age = plans["age_condition"].fillna("전연령").value_counts().head(7)[::-1]
axes[1].barh(age.index, age.values, color=C["blue"], height=0.65)
axes[1].set_xscale("log"); axes[1].set_xlabel("요금제 수 (log)")
axes[1].set_title("연령 조건별 요금제 수")
axes[1].grid(axis="y", visible=False)
show("segment_composition")

**해석.** 두 분포는 거의 겹치지 않는다. 같은 테이블에 두고 가격 기준으로 정렬하면
**MNO는 100% 탈락**하고, "MNO가 후보에 하나도 없다"는 결과가 버그처럼 보이지만 사실은 정상이다.
그래서 비교 그룹 키에 `carrier_type`을 반드시 포함해야 한다.

브랜드 집중도는 다른 얘기를 한다. 상위 5개 브랜드가 목록의 상당 부분을 차지하므로,
**정렬만 하면 Top-5가 한 브랜드로 도배**될 수 있다. 사용자 신뢰 관점에서 나쁘다.

(채널 분류 자체를 다시 보는 작업은 **13장**에서 다룬다.)

> **→ 파이프라인 반영 #5**: (a) 걸러내기·순위 계산의 그룹 키에 `carrier_type` 포함,
> (b) Top-N 구성에 **브랜드 다양성 제약**(같은 브랜드 최대 2개) 추가 — 평가지표 `candidate_diversity`와 연결.

## 6. 비교 항목 정리 — 스펙을 비교 가능한 숫자로

"데이터 11GB + 매일 2GB + 소진 후 3Mbps"는 하나의 숫자로 합칠 수 없다. **축을 나눠서 본다.**

| 축 | 원본 컬럼 | 정규화 방법 |
|---|---|---|
| 데이터 총량 | `data_gb`, `daily_data_gb`, `data_unlimited` | 월 환산 GB + 무제한 플래그(별도) |
| 데이터 다 쓴 뒤 속도 | `data_throttle_speed` | Mbps 숫자로 변환, 결측=0 (=사실상 끊김) |
| 음성 | `voice_minutes`, `voice_unlimited` | 분 + 무제한 플래그 |
| 테더링 | `tethering_gb`, `tethering_support` | GB + 지원 여부 3범주 |
| 혜택 | `ott_option_count`, 혜택 테이블 | 개수 + 9장에서 원화 환산 |

무제한을 999999 같은 큰 값으로 채우면 거리 계산과 회귀가 그 축에 끌려간다. **플래그로 분리**한다.

In [ ]:
# data_gb 가 daily 를 이미 월 환산해 담고 있는지 검증
chk = plans[plans["daily_data_gb"].notna()].copy()
chk["추정"] = chk["base_data_gb"].fillna(0) + chk["daily_data_gb"] * 30
print("일일제공 요금제", len(chk), "건 중 data_gb == base + daily*30 인 비율:",
      f"{np.isclose(chk['data_gb'], chk['추정']).mean():.1%}")
print(chk[["plan_name", "base_data_gb", "daily_data_gb", "data_gb", "data_throttle_speed"]].head(4).to_string(index=False))

In [ ]:
def speed_to_mbps(value) -> float:
    if pd.isna(value):
        return 0.0
    t = str(value).strip().lower().replace(" ", "")
    try:
        if t.endswith("mbps"):
            return float(t.removesuffix("mbps"))
        if t.endswith("kbps"):
            return float(t.removesuffix("kbps")) / 1000
    except ValueError:
        return 0.0
    return 0.0

F = pd.DataFrame(index=plans.index)
F["plan_id"] = plans["plan_id"]
F["plan_name"] = plans["plan_name"]
F["brand"] = plans["mvno_brand"].fillna(plans["host_mno"])
F["carrier_type"] = plans["carrier_type"]
F["fee"] = plans["eff_24m"]                                   # 랭킹용 단일 가격 축
F["data_gb"] = plans["data_gb"]                               # 무제한은 NaN 유지
F["data_unlimited"] = plans["data_unlimited"].fillna(False)
F["qos_mbps"] = plans["data_throttle_speed"].map(speed_to_mbps)
F["voice_min"] = plans["voice_minutes"]
F["voice_unlimited"] = plans["voice_unlimited"].fillna(False)
F["sms_unlimited"] = plans["sms_unlimited"].fillna(False)
F["tether_gb"] = np.where(plans["tethering_support"].eq("unsupported"), 0.0, plans["tethering_gb"])
F["tether_known"] = ~plans["tethering_support"].eq("undisclosed")
F["ott_n"] = plans["ott_option_count"].fillna(0)
F["online_only"] = plans["is_online_only"].fillna(False)
F["age_cond"] = plans["age_condition"].fillna("전연령")
F["network_gen"] = plans["network_gen"].fillna("미상")

print(F.describe(include="all").T[["count", "mean", "min", "max"]].to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
d = F.loc[F["carrier_type"] == "MVNO", "data_gb"].dropna()
axes[0].hist(np.log10(d.clip(lower=0.1)), bins=40, color=C["blue"])
axes[0].set_xlabel("월 데이터 GB (로그 눈금)"); axes[0].set_ylabel("요금제 수")
axes[0].set_title("데이터 용량 분포 (로그 눈금)")
q = F.loc[F["carrier_type"] == "MVNO", "qos_mbps"]
vc = q.value_counts().sort_index()
axes[1].bar([str(v) for v in vc.index], vc.values, color=C["blue"])
axes[1].set_xlabel("다 쓴 뒤 속도 (Mbps, 0 = 미공개/없음)"); axes[1].set_ylabel("요금제 수")
axes[1].set_title("데이터를 다 쓴 뒤 속도 — 종류별 요금제 수")
show("data_qos_axes")

In [ ]:
# 축 간 상관 — 어떤 축이 서로 중복인가 (가중치 설계 전에 반드시 본다)
num = pd.DataFrame({
    "실효요금": F["fee"],
    "데이터(log)": np.log1p(F["data_gb"].fillna(F["data_gb"].max())),
    "무제한": F["data_unlimited"].astype(int),
    "다 쓴 뒤 속도": F["qos_mbps"],
    "음성무제한": F["voice_unlimited"].astype(int),
    "문자무제한": F["sms_unlimited"].astype(int),
    "테더링GB": F["tether_gb"].fillna(0),
    "OTT수": F["ott_n"],
})
corr = num.corr(method="spearman")

fig, ax = plt.subplots(figsize=(6.8, 5.8))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=9)
for i in range(len(corr)):
    for j in range(len(corr)):
        v = corr.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(v) > 0.55 else C["ink"])
fig.colorbar(im, ax=ax, shrink=0.8, label="순위 상관 (−1 ~ 1)")
ax.set_title("비교 항목끼리 얼마나 겹치는가 (순위 기준 상관)")
ax.grid(False)
show("axis_correlation")

In [ ]:
# 데이터 용량은 연속형이 아니라 '업계 표준값'의 반복이다
vc = plans["data_gb"].value_counts().head(18).sort_values()
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.barh([f"{v:g}GB" for v in vc.index], vc.values, color=C["blue"], height=0.68)
for y, v in enumerate(vc.values):
    ax.text(v + 2, y, str(v), va="center", fontsize=8.5, color=C["ink"])
ax.set_xlabel("요금제 수"); ax.set_title("가장 많이 쓰이는 데이터 용량")
ax.grid(axis="y", visible=False)
show("data_gb_modes")
print("상위 18개 용량이 전체에서 차지하는 비중:",
      f"{vc.sum()/plans['data_gb'].notna().sum():.1%}")

**해석.** 데이터 용량은 선형 축에서 보면 왼쪽에 뭉쳐 보이지만 로그 축에서는 **여러 개의 봉우리**로 갈린다
(1GB대 / 10GB대 / 100GB대 = 사실상 다른 상품군). 그래서 거리·회귀에서 `data`는 `log1p` 변환이 맞다.

다 쓴 뒤 속도는 연속된 숫자처럼 보이지만 실제 값은 **{0, 0.1, 0.4, 1, 3, 5, 10} 몇 가지**뿐이다.
그리고 무제한 요금제 중 상당수가 "다 쓴 뒤 속도" 값을 가진다 — 무제한이라고 다 같은 무제한이 아니라는 뜻이고,
이건 리포트에서 반드시 경고로 나가야 하는 항목이다.

> **→ 파이프라인 반영 #6**: `data`는 log1p, `cost`는 sqrt(가중치 산출에서 쓰는 변환과 동일),
> 다 쓴 뒤 속도는 **등급(순서형)** 으로 취급. `tethering_support == "undisclosed"`는 0이 아니라
> **확인 불가**로 따로 두고, "테더링 필요" 조건에서 무조건 빼지 말고 경고와 함께 순위를 낮춘다.

## 7. 가격 대비 성능 — 가성비 한계선

추천의 기준은 "**같은 가격이면 더 좋은 것, 같은 성능이면 더 싼 것**"이다.
그 경계선을 **가성비 한계선**(파레토 프론티어)이라고 한다. 선 안쪽 요금제는 더 싸면서 더 좋은 대안이 있다는 뜻이다.
여기서는 가격과 데이터 2개 축으로 그린다.

In [ ]:
sub = F[(F["carrier_type"] == "MVNO") & F["data_gb"].notna() & (F["fee"] > 0)].copy()

def pareto_front(cost: np.ndarray, benefit: np.ndarray) -> np.ndarray:
    # 비용 오름차순으로 훑으며 최대 성능 갱신점만 남긴다.
    order = np.lexsort((-benefit, cost))
    keep, best = np.zeros(len(cost), bool), -np.inf
    for i in order:
        if benefit[i] > best:
            keep[i] = True
            best = benefit[i]
    return keep

sub["frontier"] = pareto_front(sub["fee"].to_numpy(), sub["data_gb"].to_numpy())
print(f"MVNO 유한데이터 {len(sub):,}개 중 파레토 경계 위: {sub['frontier'].sum()}개 "
      f"({sub['frontier'].mean():.1%})")
sub.loc[sub["frontier"], ["brand", "plan_name", "fee", "data_gb", "qos_mbps"]].sort_values("fee").head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.4))
ax.scatter(sub.loc[~sub["frontier"], "fee"], sub.loc[~sub["frontier"], "data_gb"],
           s=10, alpha=0.25, color=C["grey"], edgecolors="none", label="한계선 안쪽")
fr = sub[sub["frontier"]].sort_values("fee")
ax.plot(fr["fee"], fr["data_gb"], color=C["orange"], lw=1.6, marker="o", ms=4.5,
        label="가성비 한계선")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("24개월 기준 실제 월요금 (원, 로그)"); ax.set_ylabel("월 데이터 GB (로그)")
ax.set_title("요금 대비 데이터 — 주황선 위쪽에는 상품이 없다 (가성비 한계선)")
ax.legend(frameon=False, loc="lower right")
show("pareto_frontier")

In [ ]:
# 가격-데이터 분포: 상위 5개 브랜드 강조
mv_all = F[F["carrier_type"] == "MVNO"]
top5b = mv_all["brand"].value_counts().head(5).index.tolist()
fig, ax = plt.subplots(figsize=(9, 5.6))
rest = mv_all[~mv_all["brand"].isin(top5b)]
ax.scatter(rest["fee"], rest["data_gb"].fillna(400), s=8, alpha=0.18, color=C["grey"],
           edgecolors="none", label="그 외 브랜드")
for color, b in zip([C["blue"], C["orange"], C["green"], C["pink"], C["sky"]], top5b):
    g = mv_all[mv_all["brand"] == b]
    ax.scatter(g["fee"], g["data_gb"].fillna(400), s=16, alpha=0.75, color=color, label=b,
               edgecolors="none")
ax.axhline(330, color=C["ink"], lw=0.8, ls=":")
ax.text(700, 360, "↑ 이 위쪽 = 무제한 (숫자가 없어 400 자리에 찍음)", fontsize=8.5, color=C["ink"])
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("24개월 기준 실제 월요금 (원, 로그)"); ax.set_ylabel("월 데이터 GB (로그)")
ax.set_title("브랜드별로 어느 요금·데이터 구간을 파는가")
ax.legend(frameon=False, fontsize=8.5, loc="lower right", ncol=2)
show("brand_positioning")

In [ ]:
sub["won_per_gb"] = sub["fee"] / sub["data_gb"]
bins = [0, 3, 11, 31, 71, 151, 1e9]
labels = ["~3GB", "3~11GB", "11~31GB", "31~71GB", "71~151GB", "151GB+"]
sub["구간"] = pd.cut(sub["data_gb"], bins=bins, labels=labels)
box = [sub.loc[sub["구간"] == l, "won_per_gb"].dropna().values for l in labels]

fig, ax = plt.subplots()
bp = ax.boxplot(box, labels=labels, showfliers=False, patch_artist=True, widths=0.6)
for patch in bp["boxes"]:
    patch.set_facecolor(C["blue"]); patch.set_alpha(0.65); patch.set_edgecolor(C["ink"])
for med in bp["medians"]:
    med.set_color(C["orange"]); med.set_linewidth(2)
ax.set_yscale("log"); ax.set_ylabel("1GB당 가격 (원, 로그)")
ax.set_title("1GB당 가격 — 데이터 구간별")
ax.grid(axis="x", visible=False)
show("won_per_gb")
print(sub.groupby("구간")["won_per_gb"].median().round(0).to_string())

**해석.** 두 가지가 동시에 보인다.

1. **한계선 위 요금제는 소수**다. 나머지는 "더 싸면서 데이터도 많은" 대안이 있다 → 8장으로 이어진다.
2. **단가는 구간마다 자릿수가 다르다.** 그래서 "GB당 단가가 싼 순"으로 추천하면
   **항상 대용량 요금제가 이긴다** — 데이터를 5GB만 쓰는 사람에게 100GB를 권하게 된다.

이게 단가 기준 추천의 함정이다. 필요한 건 단가가 아니라 **"내 사용량에서의 기대 청구액"** 이다.

> **→ 파이프라인 반영 #7**: 랭킹 점수에 `won_per_gb`를 직접 넣지 않는다.
> 사용량 분포(LogNormal)를 두고 **초과요금까지 포함한 기대 청구액**으로 비교한다.
> 가성비 한계선은 순위 계산이 아니라 **후보를 미리 줄이는 용도와, 리포트에서 "왜 이게 최선인가" 설명**에 쓴다.

## 8. 더 나은 대안이 있는 요금제 걸러내기

"A가 B보다 모든 항목에서 낫다"면 B는 추천할 이유가 없다. 이런 요금제를 **열위(dominated)** 라고 부른다.
관건은 **비교 항목을 몇 개로 잡느냐**다. 항목이 적으면 대부분이 걸러져 후보가 지나치게 줄고,
항목이 많으면 아무것도 걸러지지 않는다. 3개 항목과 6개 항목으로 비교한다.

In [ ]:
def dominated_flags(M: np.ndarray, chunk: int = 300):
    # M: (n,k) 클수록 좋은 값. 자신을 지배하는 다른 행이 있으면 True.
    n = M.shape[0]
    dominated = np.zeros(n, bool)
    dominator = np.full(n, -1)
    for s in range(0, n, chunk):
        blk = M[s:s + chunk]
        ge = (M[None, :, :] >= blk[:, None, :]).all(-1)
        gt = (M[None, :, :] > blk[:, None, :]).any(-1)
        dom = ge & gt
        for i in range(blk.shape[0]):
            dom[i, s + i] = False
        has = dom.any(1)
        dominated[s:s + chunk] = has
        dominator[s:s + chunk] = np.where(has, dom.argmax(1), -1)
    return dominated, dominator

G = F[F["carrier_type"] == "MVNO"].reset_index(drop=True).copy()
BIG_DATA = G["data_gb"].max() * 2
BIG_VOICE = (G["voice_min"].max() if G["voice_min"].notna().any() else 0) * 2 + 1

axes3 = np.column_stack([
    -G["fee"].to_numpy(),
    np.where(G["data_unlimited"], BIG_DATA, G["data_gb"].fillna(0)),
    np.where(G["voice_unlimited"], BIG_VOICE, G["voice_min"].fillna(0)),
])
axes6 = np.column_stack([
    axes3.T[0], axes3.T[1], axes3.T[2],
    G["qos_mbps"].to_numpy(),
    G["tether_gb"].fillna(0).to_numpy(),
    G["ott_n"].to_numpy(),
])

d3, _ = dominated_flags(axes3)
d6, who6 = dominated_flags(axes6)
print(f"3개 항목(요금·데이터·음성)으로 비교 : {d3.mean():.1%} 가 걸러짐  → {int((~d3).sum())}개 남음")
print(f"6개 항목(+속도·테더링·OTT)으로 비교  : {d6.mean():.1%} 가 걸러짐  → {int((~d6).sum())}개 남음")

In [ ]:
# 조건 그룹을 나눠 다시 계산 — 그룹을 섞으면 조건부 상품(청년·시니어 전용)이 억울하게 탈락한다
print("MVNO 중 온라인 전용 비율:", f"{G['online_only'].mean():.0%}", "← 상수라서 그룹 키로 쓸 수 없다")
surv = []
for age, g in G.groupby("age_cond"):
    if len(g) < 2:
        surv.append({"조건 그룹": age, "요금제": len(g), "생존": len(g), "생존율%": 100.0})
        continue
    A = np.column_stack([
        -g["fee"].to_numpy(),
        np.where(g["data_unlimited"], BIG_DATA, g["data_gb"].fillna(0)),
        np.where(g["voice_unlimited"], BIG_VOICE, g["voice_min"].fillna(0)),
        g["qos_mbps"].to_numpy(), g["tether_gb"].fillna(0).to_numpy(), g["ott_n"].to_numpy(),
    ])
    dd, _ = dominated_flags(A)
    surv.append({"조건 그룹": age, "요금제": len(g),
                 "생존": int((~dd).sum()), "생존율%": round((~dd).mean() * 100, 1)})
pd.DataFrame(surv).sort_values("요금제", ascending=False).head(12)

In [ ]:
# 축을 하나씩 늘릴 때 생존율이 어떻게 변하는가
axis_names = ["요금", "데이터", "음성", "속도", "테더링", "OTT"]
cols_all = [axes6[:, i] for i in range(6)]
surv_curve = []
for k in range(2, 7):
    A = np.column_stack(cols_all[:k])
    dk, _ = dominated_flags(A)
    surv_curve.append({"축 수": k, "축": " + ".join(axis_names[:k]),
                       "생존": int((~dk).sum()), "생존율%": round((~dk).mean() * 100, 2)})
sc = pd.DataFrame(surv_curve)

fig, ax = plt.subplots(figsize=(8.5, 4.4))
ax.plot(sc["축 수"], sc["생존"], marker="o", ms=8, lw=2, color=C["blue"])
for _, r in sc.iterrows():
    ax.annotate(f"{r['생존']}개\n({r['생존율%']}%)", (r["축 수"], r["생존"]),
                textcoords="offset points", xytext=(0, 12), ha="center", fontsize=9, color=C["ink"])
ax.set_xticks(sc["축 수"]); ax.set_xlabel("비교 항목 수 (요금 → +데이터 → +음성 → +다 쓴 뒤 속도 → +테더링 → +OTT)")
ax.set_ylabel("더 나은 대안이 없는 요금제 수"); ax.set_ylim(0, sc["생존"].max() * 1.35)
ax.set_title("비교 항목을 늘릴수록 살아남는 요금제 수")
show("dominance_axis_curve")
sc

In [ ]:
# 걸러지지 않고 남은 요금제의 위치
fig, ax = plt.subplots(figsize=(9, 5.4))
G_ = G.copy(); G_["survive"] = ~d6
ax.scatter(G_.loc[G_["survive"] == False, "fee"], G_.loc[G_["survive"] == False, "data_gb"].fillna(400),
           s=8, alpha=0.18, color=C["grey"], edgecolors="none", label=f"더 나은 대안 있음 ({int((~G_['survive']).sum()):,}개)")
ax.scatter(G_.loc[G_["survive"], "fee"], G_.loc[G_["survive"], "data_gb"].fillna(400),
           s=34, color=C["orange"], edgecolors="white", lw=0.6,
           label=f"남은 후보 ({int(G_['survive'].sum())}개)", zorder=4)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("24개월 기준 실제 월요금 (원, 로그)"); ax.set_ylabel("월 데이터 GB (로그, 400 = 무제한)")
ax.set_title("더 나은 대안이 있는 요금제를 걸러내고 남은 것")
ax.legend(frameon=False, loc="lower right")
show("dominance_survivors")

In [ ]:
# 왜 걸러졌는지를 사람이 읽을 수 있게 — 밀린 상대 표시
ex = np.where(d6)[0][:5]
rows = []
for i in ex:
    j = who6[i]
    rows.append({
        "걸러진 요금제": G.loc[i, "plan_name"][:28], "요금": round(G.loc[i, "fee"]),
        "데이터GB": G.loc[i, "data_gb"],
        "더 나은 대안": G.loc[j, "plan_name"][:28],
        "대안 요금": round(G.loc[j, "fee"]), "대안 데이터GB": G.loc[j, "data_gb"],
    })
pd.DataFrame(rows)

**해석.** 요금·데이터·음성 3개만 보면 거의 전부가 걸러진다. 계산은 맞지만 **실제 선택 기준과 다르다** —
"이건 테더링이 되고 저건 안 된다" 같은 구매 이유가 비교 항목에 없기 때문이다.
6개 항목으로 늘리고 **조건 그룹(연령 전용 등)을 분리**하면 남는 후보가 현실적인 수준이 된다.

그리고 이 작업의 진짜 쓸모는 후보를 줄이는 것보다 **설명**에 있다.
아래 표처럼 "이 요금제는 왜 추천에 안 나왔나"에 한 줄로 답할 수 있다.

> **→ 파이프라인 반영 #8**: 걸러내기는 **6개 항목 + (carrier_type, online_only, age_cond) 그룹** 기준.
> 어떤 요금제에 밀렸는지를 `dominated_by` 컬럼으로 남겨 리포트의 "제외 사유"로 쓴다.

## 9. 혜택·OTT를 금액으로 환산

"넷플릭스 포함 3만원"과 "혜택 없는 3만원"은 같은 가격이 아니다.
혜택을 금액으로 환산해 **실질 요금(net effective price)** 을 만들고, 순위가 바뀌는지 확인한다.
다만 **환산값은 과대평가되기 쉽다.** 사용하지 않는 OTT는 가치가 0이므로 상한과 감가율을 둔다.

In [ ]:
print("혜택 테이블 커버리지:", f"{benefits['plan_id'].nunique():,}개 요금제 / 전체 {len(plans):,}")
print(benefits["benefit_category"].value_counts().to_string())

price_map = dict(zip(ott_prices["service"], ott_prices["monthly_won"]))

def ott_value(row, utilization: float = 0.6, cap: int = 20000) -> float:
    # OTT 옵션 문자열을 원화 가치로. utilization = '실제로 쓰는 비율' 감가.
    if pd.isna(row):
        return 0.0
    total = 0
    for token in str(row).split("|"):
        t = token.strip()
        for svc, won in price_map.items():
            if svc in t or t in svc:
                total += won
                break
    return min(total * utilization, cap)

plans["ott_value_won"] = plans["ott_options"].map(ott_value)
plans["net_eff_24m"] = plans["eff_24m"] - plans["ott_value_won"]

has_ott = plans[plans["ott_value_won"] > 0]
print(f"\nOTT 가치가 잡힌 요금제: {len(has_ott):,}개 | 중앙값 {has_ott['ott_value_won'].median():,.0f}원/월")
print(plans.groupby("carrier_type")["ott_value_won"].mean().round(0).to_string())

In [ ]:
mv2 = plans[plans["carrier_type"] == "MVNO"].copy()
mno2 = plans[plans["carrier_type"] == "MNO"].copy()

for label, d_ in [("MVNO", mv2), ("MNO", mno2)]:
    d_["rank_shift"] = d_["net_eff_24m"].rank() - d_["eff_24m"].rank()   # 음수 = 순위 상승
    b = set(d_.nsmallest(20, "eff_24m")["plan_id"])
    a = set(d_.nsmallest(20, "net_eff_24m")["plan_id"])
    print(f"[{label}] 최저가 Top-20 겹침 {len(a & b)}/20 | "
          f"20계단 이상 이동 {(d_['rank_shift'].abs() >= 20).mean():.1%} | "
          f"최대 상승 {abs(d_['rank_shift'].min()):.0f}계단")

print("\n순위가 가장 많이 오른 MNO 요금제 (OTT 가치 차감 효과)")
print(mno2.nsmallest(6, "rank_shift")[
    ["plan_name", "monthly_fee", "eff_24m", "ott_value_won", "net_eff_24m", "rank_shift"]
].to_string(index=False))

fig, ax = plt.subplots()
ax.scatter(mno2["eff_24m"], mno2["net_eff_24m"], s=14, alpha=0.5, color=C["orange"], label="MNO")
ax.scatter(mv2["eff_24m"], mv2["net_eff_24m"], s=8, alpha=0.25, color=C["blue"], label="MVNO")
lim = [0, plans["eff_24m"].max() * 1.02]
ax.plot(lim, lim, color=C["grey"], lw=1.1, ls="--")
ax.set_xlabel("24개월 기준 실제 월요금 (원)"); ax.set_ylabel("OTT 값어치를 뺀 요금 (원)")
ax.set_title("OTT 값어치를 뺀 실제 부담 요금")
ax.legend(frameon=False, loc="upper left")
show("ott_shift")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
hv = plans.loc[plans["ott_value_won"] > 0, "ott_value_won"]
axes[0].hist(hv, bins=30, color=C["blue"])
axes[0].set_xlabel("OTT 값어치 환산액 (원/월)"); axes[0].set_ylabel("요금제 수")
axes[0].set_title(f"요금제에 딸린 OTT 값어치 ({len(hv)}개)")

for color, (label, d_) in zip([C["blue"], C["orange"]], [("MVNO", mv2), ("MNO", mno2)]):
    axes[1].hist(d_["rank_shift"], bins=40, alpha=0.7, color=color, label=label)
axes[1].axvline(0, color=C["ink"], lw=1)
axes[1].set_xlabel("순위 변화 (음수 = 순위 상승)"); axes[1].set_ylabel("요금제 수")
axes[1].set_title("OTT를 반영하면 순위가 얼마나 움직이나")
axes[1].legend(frameon=False)
show("ott_value_and_shift")

**해석 (예상과 다른 결과가 나오는 곳).**
**최저가 Top-20은 거의 그대로다.** 당연하다 — 제일 싼 요금제에는 애초에 번들이 없다.
OTT 보정이 흔드는 건 최저가 구간이 아니라 **중·고가 구간의 상대 순위**이고,
평균 OTT 가치가 MVNO 410원 대 MNO 5,928원인 데서 보이듯 **효과는 MNO 쪽에 몰려 있다**.

이 구분이 중요한 이유: "OTT 보정하면 추천이 바뀐다"고 뭉뚱그리면 틀린 말이다.
정확히는 **"가성비 최저가를 찾는 사용자에겐 영향이 없고, 번들을 이미 쓰는 사용자에겐 크게 바뀐다."**
즉 이 기능의 대상 사용자는 따로 있다.

다만 `utilization=0.6`, `cap=20000`은 **가정**이다. 가정이 결과를 만든 곳은 반드시 표시해야 한다.
서비스에서는 "이미 넷플릭스를 구독 중인가?"를 프로파일링에서 물어보고 **개인별로 가치를 계산**하는 게 맞다.

> **→ 파이프라인 반영 #9**: OTT 가치는 전역 상수가 아니라 `UserProfile.subscribed_services` 기반 개인화.
> 사용자가 안 쓰는 OTT는 가치 0. 리포트에는 "OTT 가치 X원 차감 전/후" 두 숫자를 모두 노출.

## 10. 가입자 수 분포 — 가중치 근거로 쓸 수 있나

개인별 선택 로그가 없고 **요금제별 가입자 수(집계)** 만 있다. 가중치 근거로 쓰려면
먼저 **분포 형태**를 확인해야 한다. 소수에 쏠려 있으면 회귀 결과가 상위 몇 개에 끌려간다.

In [ ]:
s = plans.loc[plans["subscriber_count"].notna(), "subscriber_count"]
print(s.describe().round(0).to_string())
print(f"\n상위 1% 요금제가 차지하는 가입자 비중: {s.nlargest(max(1,int(len(s)*0.01))).sum()/s.sum():.1%}")
print(f"상위 10%: {s.nlargest(int(len(s)*0.1)).sum()/s.sum():.1%}")
print("가입자 수 결측(=MNO 등):", plans['subscriber_count'].isna().sum())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].hist(s, bins=60, color=C["blue"])
axes[0].set_title("가입자 수 분포 — 일반 눈금")
axes[0].set_xlabel("가입자 수"); axes[0].set_ylabel("요금제 수")
axes[1].hist(np.log10(s), bins=40, color=C["blue"])
axes[1].set_title("가입자 수 분포 — 로그 눈금")
axes[1].set_xlabel("가입자 수 (로그 눈금)")
show("subscriber_scale")

In [ ]:
# 가입자 쏠림을 한 장으로
ss = np.sort(s.values)
cum = np.cumsum(ss) / ss.sum()
x = np.arange(1, len(ss) + 1) / len(ss)
fig, ax = plt.subplots(figsize=(5.8, 5.4))
ax.plot(x, cum, color=C["blue"], lw=2)
ax.plot([0, 1], [0, 1], color=C["grey"], lw=1.2, ls="--")
ax.fill_between(x, cum, x, color=C["blue"], alpha=0.12)
gini = 1 - 2 * float(np.sum((cum[:-1] + cum[1:]) / 2 * np.diff(x)))
ax.text(0.05, 0.86, f"지니계수 ≈ {gini:.2f}", fontsize=11, color=C["blue"])
ax.set_xlabel("요금제를 가입자 적은 순으로 줄 세웠을 때 누적 비율"); ax.set_ylabel("가입자 누적 비율")
ax.set_title("가입자 쏠림 정도 — 대각선에서 멀수록 일부 요금제에 집중")
show("subscriber_lorenz")

In [ ]:
mv3 = plans[plans["carrier_type"] == "MVNO"].dropna(subset=["subscriber_count"]).copy()
mv3["share"] = mv3["subscriber_count"] / mv3["subscriber_count"].sum()
mv3["log_share"] = np.log(mv3["share"])
mv3["log_data"] = np.log1p(mv3["data_gb"].fillna(mv3["data_gb"].max()))

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.scatter(mv3["eff_24m"], mv3["log_share"], s=9, alpha=0.3, color=C["blue"], edgecolors="none")
coef = np.polyfit(mv3["eff_24m"], mv3["log_share"], 1)
xs = np.linspace(mv3["eff_24m"].min(), mv3["eff_24m"].max(), 50)
ax.plot(xs, np.polyval(coef, xs), color=C["orange"], lw=2)
ax.set_xlabel("24개월 기준 실제 월요금 (원)"); ax.set_ylabel("가입자 점유율 (로그)")
ax.set_title("요금과 가입자 점유율의 관계")
ax.text(0.98, 0.94, f"기울기 {coef[0]:.2e} / 원", transform=ax.transAxes,
        ha="right", fontsize=10, color=C["orange"])
show("price_vs_logshare")

r2 = np.corrcoef(mv3["eff_24m"], mv3["log_share"])[0, 1] ** 2
print(f"가격 단독 회귀 R² = {r2:.4f}   ← 거의 0이다")

print("\n가입자 점유율(로그)과 각 항목의 순위 상관")
cand = pd.DataFrame({
    "실효요금": mv3["eff_24m"],
    "데이터(log)": mv3["log_data"],
    "다 쓴 뒤 속도": mv3["data_throttle_speed"].map(speed_to_mbps),
    "음성무제한": mv3["voice_unlimited"].astype(int),
    "OTT 개수": mv3["ott_option_count"].fillna(0),
    "혜택 개수": mv3["benefit_count"].fillna(0),
    "log_share": mv3["log_share"],
})
print(cand.corr(method="spearman")["log_share"].drop("log_share").round(3).sort_values().to_string())

print("\n브랜드별 가입자 합계 상위 8 (노출 편향의 증거)")
print(mv3.groupby("mvno_brand")["subscriber_count"].sum().sort_values(ascending=False).head(8).to_string())
print("\n브랜드 평균 log_share 의 분산이 전체 분산에서 차지하는 비율:",
      f"{mv3.groupby('mvno_brand')['log_share'].transform('mean').var() / mv3['log_share'].var():.1%}")

In [ ]:
# 가격 × 데이터 × 가입자 버블 — 실제로 '팔리는 자리'는 어디인가
mv4 = mv3.dropna(subset=["data_gb"])
fig, ax = plt.subplots(figsize=(9, 5.6))
sizes = np.sqrt(mv4["subscriber_count"]) * 1.1
sc_ = ax.scatter(mv4["eff_24m"], mv4["data_gb"], s=sizes, alpha=0.35,
                 color=C["blue"], edgecolors="none")
big = mv4.nlargest(6, "subscriber_count")
ax.scatter(big["eff_24m"], big["data_gb"], s=np.sqrt(big["subscriber_count"]) * 1.1,
           facecolor="none", edgecolors=C["red"], lw=1.6, zorder=5)
for _, r in big.iterrows():
    ax.annotate(f"{r['plan_name'][:16]}\n{r['subscriber_count']:,.0f}명",
                (r["eff_24m"], r["data_gb"]), textcoords="offset points", xytext=(8, 8),
                fontsize=8, color=C["red"])
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("24개월 기준 실제 월요금 (원, 로그)"); ax.set_ylabel("월 데이터 GB (로그)")
ax.set_title("실제로 가입자가 몰리는 구간 (원 크기 = 가입자 수)")
show("subscriber_bubble")

**해석 — 결과를 과대 해석하면 안 되는 구간이다.**

1. 가입자 수는 **소수 요금제에 심하게 쏠려 있다**(상위 10%가 대부분을 차지). 그래서 회귀의 종속변수는
   점유율을 로그로 바꾼 값(`log_share`)을 쓴다. 그냥 쓰면 상위 몇 개에 결과가 끌려간다.
2. 그런데 **가격 단독 R²는 0.01 수준**이다. "싼 게 많이 팔린다"는 직관이 이 데이터에서는 거의 설명력이 없다.
   즉 **가격 계수 하나만 뽑아 "사용자는 가격을 제일 중시한다"고 말하면 안 된다.**
3. 브랜드 평균만으로도 log_share 분산의 상당 부분이 설명된다 = **노출·유통 효과가 섞여 있다.**
   모요 상단 노출, 제휴 프로모션, 브랜드 인지도는 "선호"가 아니다.

그래서 집계 점유율 역산(Berry 방식)은 쓸 수 있지만, **계수 = 선호 가중치**라는 등식은 성립하지 않는다.
멘토가 지적한 "선택 수 회귀 계수를 그대로 추천 가중치로 쓰는 게 맞나"가 정확히 이 지점이고,
이 노트북의 답은 **"브랜드 고정효과를 넣기 전까지는 아니다"** 이다.

> **→ 파이프라인 반영 #10**: 가중치는 `log_share` 선형회귀의 표준화 계수로 유지하되,
> **브랜드 고정효과**를 넣어 노출 편향을 흡수한 버전과 비교해 민감도를 보고한다.

## 11. 시간에 따른 변화 — 요금제가 사라지는 속도

카탈로그는 특정 시점 스냅샷이지만, `crawler/data/review/changes_*.csv` 에 **일자별 변경 로그**가 쌓여 있다.
여기서 **단종 위험** 축을 만들 수 있다.

In [ ]:
paths = sorted(glob.glob(str(ROOT / "crawler/data/review/changes_*.csv")))
print(f"변경 로그 {len(paths)}개")
if paths:
    ch = pd.concat([pd.read_csv(p, encoding="utf-8-sig").assign(날짜=Path(p).stem.split("_")[-1])
                    for p in paths], ignore_index=True)
    print(ch["change_type"].value_counts().to_string())
    piv = ch.pivot_table(index="날짜", columns="change_type", values="plan_id", aggfunc="count").fillna(0)
    display(piv)
    if "변경" in ch["change_type"].values:
        print("\n가장 자주 바뀌는 필드 Top 8")
        print(ch.loc[ch["change_type"] == "변경", "field"].value_counts().head(8).to_string())
else:
    print("변경 로그가 없으면 이 섹션은 건너뛴다.")

In [ ]:
if paths:
    fig, ax = plt.subplots()
    palette = [C["green"], C["red"], C["blue"], C["orange"], C["pink"]]
    order = list(piv.columns)
    colors = {c: palette[i % len(palette)] for i, c in enumerate(order)}
    bottom = np.zeros(len(piv))
    for c in order:
        ax.bar(piv.index, piv[c].values, bottom=bottom, color=colors[c], label=c, width=0.6)
        bottom += piv[c].values
    ax.set_ylabel("건수"); ax.legend(frameon=False)
    ax.set_title("수집 회차마다 새로 생기고 없어진 요금제 수")
    ax.grid(axis="x", visible=False)
    show("catalog_changes")

In [ ]:
if paths and "변경" in ch["change_type"].values:
    ff = ch.loc[ch["change_type"] == "변경", "field"].value_counts().head(10)[::-1]
    fig, ax = plt.subplots(figsize=(8.5, 4.4))
    colors_f = [C["red"] if f in ("discounted_fee", "monthly_fee", "discount_period_months")
                else C["blue"] for f in ff.index]
    ax.barh(ff.index, ff.values, color=colors_f, height=0.66)
    for y, v in enumerate(ff.values):
        ax.text(v + 4, y, str(v), va="center", fontsize=9, color=C["ink"])
    ax.set_xlabel("변경 건수"); ax.set_title("무엇이 바뀌는가 (빨강 = 가격 관련 항목)")
    ax.grid(axis="y", visible=False)
    show("changed_fields")
    price_fields = ["discounted_fee", "monthly_fee", "discount_period_months", "discount_type"]
    sh = ch[ch["change_type"] == "변경"]["field"].isin(price_fields).mean()
    print(f"변경 건 중 가격 관련 필드 비중: {sh:.1%}  ← 캐시한 가격은 금방 낡는다")

**해석.** 회차마다 수백 건이 신규/삭제된다 = **알뜰폰 카탈로그는 몇 주 단위로 갈린다.**
"오늘 1위"가 두 달 뒤에도 존재한다고 말할 수 없다는 뜻이다.

주의: 관측 신뢰 구간은 **2026-08-11 이후**부터다. 그 이전은 추정 백필이므로 생존 분석에 넣으면 안 된다.

> **→ 파이프라인 반영 #11**: `plan_id`별 **관측 생존일수 + 최근 변경 유무**로 `discontinuation_risk` 산출,
> 위험 높은 요금제는 Top-N에서 강등하거나 "단기 프로모션" 배지. 백필 구간은 계산에서 제외.

## 12. 가상 사용자로 확인 — 앞의 결정이 추천 결과를 바꾸는가

마지막으로 **"그래서 추천 결과가 달라지는가"** 를 확인한다.
결과가 달라지지 않는 결정은 문서에만 남기고 코드에는 넣지 않아도 된다.

In [ ]:
USER_TYPES = {
    "데이터 폭식 (월 80GB, 통화 적음)": dict(data=80, voice=100, tether=0, unlimited_ok=True),
    "알뜰 최소 (월 5GB, 통화 무제한)": dict(data=5, voice="unlimited", tether=0, unlimited_ok=True),
    "노트북 테더링 (월 30GB, 테더 20GB)": dict(data=30, voice=200, tether=20, unlimited_ok=True),
    "영상 시청 (무제한 + 다 쓴 뒤 3Mbps 이상)": dict(data=0, voice=100, tether=0, unlimited_ok=True, qos=3),
}

def hard_filter(df: pd.DataFrame, spec: dict) -> pd.DataFrame:
    m = df["carrier_type"].eq("MVNO")
    m &= at_least(df["data_gb"], df["data_unlimited"], spec.get("data", 0))
    if spec.get("voice") == "unlimited":
        m &= df["voice_unlimited"].fillna(False)
    else:
        m &= at_least(df["voice_min"], df["voice_unlimited"], spec.get("voice", 0))
    if spec.get("tether", 0) > 0:
        m &= df["tether_gb"].fillna(0) >= spec["tether"]
    if spec.get("qos"):
        m &= df["qos_mbps"] >= spec["qos"]
    return df[m]

summary = []
tops = {}
for name, spec in USER_TYPES.items():
    cand = hard_filter(F, spec)
    top5 = cand.nsmallest(5, "fee")[["brand", "plan_name", "fee", "data_gb", "qos_mbps"]]
    tops[name] = top5
    summary.append({"사용 유형": name, "후보 수": len(cand),
                    "Top1 요금": round(top5["fee"].iloc[0]) if len(top5) else None,
                    "Top5 브랜드 수": top5["brand"].nunique() if len(top5) else 0})
pd.DataFrame(summary)

In [ ]:
for name, t in tops.items():
    print("■", name)
    print(t.to_string(index=False), "\n")

In [ ]:
# 사용 유형별 Top-5 가격 비교
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax_, (name, t) in zip(axes.ravel(), tops.items()):
    t2 = t.head(5)[::-1]
    ypos = np.arange(len(t2))
    ax_.barh(ypos, t2["fee"], color=C["blue"], height=0.62)
    ax_.set_yticks(ypos)
    ax_.set_yticklabels([f"{b[:8]} {n[:14]}" for b, n in zip(t2["brand"], t2["plan_name"])])
    for y, (v, g) in zip(ypos, zip(t2["fee"], t2["data_gb"])):
        ax_.text(v * 1.02, y, f"{v:,.0f}원 / {g:g}GB" if pd.notna(g) else f"{v:,.0f}원 / 무제한",
                 va="center", fontsize=8, color=C["ink"])
    ax_.set_title(name, fontsize=10.5)
    ax_.set_xlim(0, t2["fee"].max() * 1.45)
    ax_.tick_params(labelsize=8)
    ax_.grid(axis="y", visible=False)
fig.suptitle("사용 유형별 추천 Top-5 (24개월 기준 실제 요금)", x=0.02, ha="left", fontsize=12)
show("persona_top5")

In [ ]:
# 민감도: 4장의 가격 정의를 바꾸면 Top-5가 얼마나 흔들리나
alt = F.copy()
alt["fee"] = plans["discounted_fee"].values          # 표시가로 랭킹했을 때
rows = []
for name, spec in USER_TYPES.items():
    a = set(hard_filter(F, spec).nsmallest(5, "fee")["plan_id"])
    b = set(hard_filter(alt, spec).nsmallest(5, "fee")["plan_id"])
    rows.append({"사용 유형": name, "24개월 기준 Top5 중 유지": len(a & b), "총": 5})
pd.DataFrame(rows)

**해석.** 마지막 표가 이 노트북의 요약이다.
가격 정의 하나만 바꿔도 Top-5가 몇 자리 뒤집힌다면, **4장의 결정은 코드로 옮길 가치가 있다.**
반대로 겹침이 5/5면 그 결정은 (적어도 이 유형에서는) 순위에 영향이 없다는 뜻이므로
"정확하지만 중요하지 않은" 항목으로 분류하면 된다.

후보 수도 같이 본다. 후보가 3개 이하로 떨어지는 유형은 **조건을 풀어줘야 하는 구간**이다.

In [ ]:
sens = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8.5, 4))
y = np.arange(len(sens))[::-1]
ax.barh(y, [5] * len(sens), color="#E8EAED", height=0.55)
ax.barh(y, sens["24개월 기준 Top5 중 유지"], color=C["red"], height=0.55)
for yi, (n, v) in zip(y, zip(sens["사용 유형"], sens["24개월 기준 Top5 중 유지"])):
    ax.text(5.1, yi, f"5개 중 {v}개만 동일", va="center", fontsize=9.5, color=C["ink"])
ax.set_yticks(y); ax.set_yticklabels([n[:22] for n in sens["사용 유형"]], fontsize=9)
ax.set_xlim(0, 8.5); ax.set_xticks(range(6))
ax.set_xlabel("보이는 요금 기준 Top-5와 겹치는 개수")
ax.set_title("요금 기준을 바꾸면 추천 Top-5가 얼마나 유지되는가")
ax.grid(axis="y", visible=False)
show("sensitivity_top5")

## 13. 채널 재분류와 요금제 그룹

앞 장까지는 데이터가 준 분류(`carrier_type`)를 그대로 썼다. 여기서는 그 분류 자체를 의심한다.
두 가지 문제가 있고, 둘 다 **행을 세는 방식**의 문제다.

**① 너겟·요고·다이렉트의 분류.**
통신 3사가 직접 파는 온라인 전용 요금제다. 대리점 요금제와 달리 약정·결합 없이 가입하고,
모요에도 같이 올라와 있다. 성격상 **대리점 요금제보다 알뜰폰 쪽에 가깝다.**

**② 통신 3사는 메인 혜택이 바뀌면 요금제 이름이 바뀐다.**
`초이스110 (넷플릭스)` 와 `초이스110 (디즈니+)` 는 요금도 데이터도 같은 **한 요금제**인데 행이 둘이다.
행을 그대로 세면 통계와 후보 목록이 왜곡된다.

복잡한 처리는 하지 않는다. **채널 컬럼 하나 + 그룹 키 하나**만 추가해서 무엇이 달라지는지 본다.

In [ ]:
# ① 채널 3분류 — 판매 경로로 나눈다
plans["channel"] = np.where(
    plans["carrier_type"] == "MVNO", "MVNO(알뜰폰)",
    np.where(plans["is_online_only"], "온라인 직판(너겟·요고·다이렉트)", "통신3사 대리점"))

print(plans["channel"].value_counts().to_string())
print("\n채널별 정상 월요금")
print(plans.groupby("channel")["monthly_fee"].agg(["count", "min", "median", "max"]).to_string())
print("\n온라인 직판 내역")
print(plans[plans["channel"].str.startswith("온라인")]["plan_category"].value_counts().to_string())

In [ ]:
order_ch = ["MVNO(알뜰폰)", "온라인 직판(너겟·요고·다이렉트)", "통신3사 대리점"]
colors_ch = [C["blue"], C["green"], C["orange"]]

fig, ax = plt.subplots(figsize=(9, 4.6))
for i, (ch, col) in enumerate(zip(order_ch, colors_ch)):
    v = plans.loc[plans["channel"] == ch, "monthly_fee"]
    y = np.random.default_rng(0).normal(i, 0.11, len(v))
    ax.scatter(v, y, s=7, alpha=0.25, color=col, edgecolors="none")
    ax.scatter([v.median()], [i], s=130, color=col, marker="|", lw=3)
    ax.text(v.median(), i + 0.26, f"중앙값 {v.median():,.0f}원", ha="center", fontsize=9, color=col)
ax.set_yticks(range(3)); ax.set_yticklabels([c.replace("(", "\n(") for c in order_ch], fontsize=9)
ax.set_xlabel("정상 월요금 (원)")
ax.set_title("판매 경로별 요금대")
ax.grid(axis="y", visible=False)
show("channel_price")

In [ ]:
# 온라인 직판이 모요에도 올라와 있는가 — 같은 이름으로 두 행 존재
on = plans[plans["channel"].str.startswith("온라인")]
mv_ = plans[plans["channel"] == "MVNO(알뜰폰)"]
dup_names = sorted(set(on["plan_name"]) & set(mv_["plan_name"]))
print(f"온라인 직판 {len(on)}건 중 모요에도 있는 것: {len(dup_names)}건")

cmp_ = (on[on["plan_name"].isin(dup_names)][["plan_name", "monthly_fee", "discounted_fee"]]
        .rename(columns={"monthly_fee": "공식_정상가", "discounted_fee": "공식_할인가"})
        .merge(mv_[mv_["plan_name"].isin(dup_names)]
               [["plan_name", "monthly_fee", "discounted_fee", "discount_type", "subscriber_count"]]
               .rename(columns={"monthly_fee": "모요_정상가", "discounted_fee": "모요_할인가"}),
               on="plan_name"))
cmp_["모요가_더_쌈"] = cmp_["모요_할인가"] < cmp_["공식_할인가"]
print(f"그중 모요가 실제로 더 싼 것: {int(cmp_['모요가_더_쌈'].sum())}건 "
      f"(4-B장의 정상가 오염 건 제외하면 {int(cmp_.loc[cmp_['plan_name'] != '너겟49', '모요가_더_쌈'].sum())}건)")
print(f"온라인 직판 공식 행의 가입자 수 결측: {on['subscriber_count'].isna().sum()} / {len(on)}  ← 가입자 정보는 모요 행에만 있다")
cmp_[cmp_["모요가_더_쌈"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.4))
cc = cmp_.sort_values("공식_할인가").reset_index(drop=True)
for i, r in cc.iterrows():
    same = not r["모요가_더_쌈"]
    ax.plot([r["공식_할인가"], r["모요_할인가"]], [i, i],
            color=C["grey"] if same else C["red"], lw=1.6, zorder=2)
    ax.scatter([r["공식_할인가"]], [i], s=26, color=C["ink"], zorder=3)
    ax.scatter([r["모요_할인가"]], [i], s=30,
               color=C["grey"] if same else C["red"], zorder=3)
    if not same:
        ax.annotate(f'{r["plan_name"]}  {r["모요_할인가"]:,.0f}원', (r["공식_할인가"], i),
                    textcoords="offset points", xytext=(11, 0), ha="left",
                    fontsize=8.5, color=C["red"], va="center")
ax.set_yticks([]); ax.set_xlabel("실제 가입가 (원/월)"); ax.set_xlim(0, 95000)
ax.set_ylabel(f"양쪽에 모두 있는 {len(cc)}개 요금제")
ax.set_title("같은 요금제의 통신사 공식 가격 vs 모요 가격")
ax.grid(axis="y", visible=False)
show("online_direct_dup")

**①의 결론 — 가설과 데이터가 어긋나는 부분이 있다.**

"모요에서만 할인이 들어간다"는 이 스냅샷에서는 **거의 성립하지 않는다.**
양쪽에 다 있는 38건 중 가격이 실제로 다른 건 **`0 청년 다이렉트` 2건**뿐이고,
나머지는 통신사 공식가와 모요가가 같다. (`너겟49`는 4-B장에서 본 정상가 오염이라 별도 건이다.)

그래도 **재분류 자체는 맞다.** 근거는 할인이 아니라 구조다.
온라인 직판은 약정·결합 없이 가입하고 가격대도 대리점과 알뜰폰 사이에 있어서,
"대리점 요금제 = 비교 기준선"이라는 전제가 이 90건에는 적용되지 않는다.

그리고 **중복 행은 지우지 말고 합쳐야 한다.** 각 행이 가진 정보가 다르기 때문이다.

| 필드 | 어느 행을 믿을 것인가 | 이유 |
|---|---|---|
| 정상가 | **통신사 공식 행** | 모요 행은 페이백이 정상가 자리에 들어가는 사고가 있다(너겟49) |
| 할인가·프로모션 | **모요 행** | 모요 전용 프로모션은 공식 행에 없다 |
| 가입자 수 | **모요 행** | 공식 행은 90건 전부 결측이다 |

> **→ 파이프라인 반영 #13-A**: `channel` 컬럼을 추가하고 **추천 후보 = MVNO + 온라인 직판**,
> **절감 기준선 = 통신3사 대리점**으로 분리. 중복 38건은 위 표대로 필드 단위 병합(행 삭제 아님).

In [ ]:
# ② 요금제 그룹 — 메인 혜택만 다른 행을 하나로 묶는다
is_mno_side = plans["carrier_type"] == "MNO"
plans["plan_family"] = np.where(
    is_mno_side,
    plans["host_mno"] + " · " + plans["plan_name"].str.split("(").str[0].str.strip(),
    plans["mvno_brand"].fillna("") + " · " + plans["plan_name"])          # MVNO 이름은 건드리지 않는다
plans["main_benefit"] = plans["plan_name"].str.extract(r"\((.+)\)\s*$")[0].where(is_mno_side)

mno_side = plans[is_mno_side]
fam = mno_side.groupby(["channel", "plan_family"]).agg(
    행수=("plan_id", "size"), 정상가=("monthly_fee", "first"),
    요금종류=("monthly_fee", "nunique"), 혜택종류=("main_benefit", "nunique")).reset_index()

print(f"통신3사 쪽 {len(mno_side)}행  →  {len(fam)}개 요금제 그룹")
print(f"그룹 안에서 요금이 전부 같은 비율: {(fam['요금종류'] == 1).mean():.1%}")
print(f"상위 3개 그룹이 차지하는 행 비중: {fam.nlargest(3, '행수')['행수'].sum() / len(mno_side):.1%}")
print()
print("행이 가장 많은 그룹")
print(fam.nlargest(6, "행수")[["channel", "plan_family", "행수", "정상가", "혜택종류"]].to_string(index=False))

In [ ]:
print("예시 — KT 초이스110 그룹 (요금·데이터 동일, 메인 혜택만 다름)")
ex = mno_side[mno_side["plan_family"].str.endswith("초이스110")]
print(ex[["plan_name", "main_benefit", "monthly_fee", "discounted_fee", "data_unlimited", "ott_options"]]
      .drop_duplicates("plan_name").head(10).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

big = fam.nlargest(8, "행수").sort_values("행수")
axes[0].barh([f"{n.split(' · ')[1][:14]}" for n in big["plan_family"]], big["행수"],
             color=C["orange"], height=0.64)
for y, v in enumerate(big["행수"]):
    axes[0].text(v + 0.6, y, str(v), va="center", fontsize=9, color=C["ink"])
axes[0].set_xlabel("행 수"); axes[0].set_title("같은 요금제가 몇 줄로 나뉘어 있나")
axes[0].grid(axis="y", visible=False)

row_med = mno_side.groupby("channel")["monthly_fee"].median()
fam_med = fam.groupby("channel")["정상가"].median()
chs = list(row_med.index)
x = np.arange(len(chs))
axes[1].bar(x - 0.19, row_med.values / 1000, width=0.36, color=C["grey"], label="행 기준")
axes[1].bar(x + 0.19, fam_med.values / 1000, width=0.36, color=C["blue"], label="그룹 기준")
for xi, (a, b) in enumerate(zip(row_med.values, fam_med.values)):
    axes[1].text(xi - 0.19, a / 1000 + 1.5, f"{a/1000:.0f}", ha="center", fontsize=9, color=C["ink"])
    axes[1].text(xi + 0.19, b / 1000 + 1.5, f"{b/1000:.0f}", ha="center", fontsize=9, color=C["blue"])
axes[1].set_xticks(x); axes[1].set_xticklabels([c.replace("(", "\n(") for c in chs], fontsize=9)
axes[1].set_ylabel("정상 월요금 중앙값 (천원)")
axes[1].set_title("줄 단위로 셀 때 vs 요금제 단위로 셀 때")
axes[1].legend(frameon=False); axes[1].grid(axis="x", visible=False)
show("plan_family")

**②의 결론.** 통신 3사 쪽 556행은 실제로 **122개 요금제**다. 그리고 쪼개진 행은 **고가 요금제에 몰려 있다**
(초이스110·90·130 세 그룹이 150행을 차지). 그래서 행 단위로 중앙값을 내면 대리점 요금이
**90,000원**이지만, 그룹 단위로 세면 **58,000원**이다.

"MNO는 비싸다"는 우리 서비스의 핵심 메시지인데, **그 근거 숫자가 집계 단위 때문에 부풀려져 있었다.**
절감액을 언급할 거라면 이 숫자부터 그룹 기준으로 바꿔야 한다.

추천 화면에서도 같은 구조가 필요하다. 사용자에게 `초이스110`을 10장의 카드로 보여줄 이유가 없다.

```
카드 1장:  초이스110 · 110,000원 · 데이터 무제한
           메인 혜택 선택 : [유튜브 프리미엄] [넷플릭스] [디즈니+] [티빙] [폰케어] …
```

9장의 OTT 개인화와 이어진다. 사용자가 이미 넷플릭스를 본다면 **그룹은 하나로 추천하고
그 안에서 넷플릭스 변형을 고르면 된다.** 즉 혜택 변형은 랭킹 문제가 아니라 **선택지 문제**다.

> **→ 파이프라인 반영 #13-B**: `plan_family` + `main_benefit` 컬럼 추가.
> 통계·후보 목록·걸러내기는 **그룹 대표 1건**으로, 화면과 리포트는 **그룹 카드 1장 + 혜택 선택지**로.
> MVNO 이름은 프로모션 태그(`[모요only]` 등)가 섞여 있어 같은 규칙을 쓰면 안 된다 — 통신 3사에만 적용.

In [ ]:
# 두 가지를 모두 적용하면 앞 장의 숫자가 이렇게 바뀐다
before_pool = int((plans["carrier_type"] == "MVNO").sum())
after_pool = int((plans["channel"] != "통신3사 대리점").sum()) - len(dup_names)   # 중복 병합
base_rows = int((plans["channel"] == "통신3사 대리점").sum())
base_fams = int((fam["channel"] == "통신3사 대리점").sum())

summary_fix = pd.DataFrame([
    {"항목": "추천 후보 풀", "지금": f"{before_pool:,}개 (MVNO만)",
     "재분류 후": f"{after_pool:,}개 (+온라인 직판, 중복 {len(dup_names)}건 병합)"},
    {"항목": "절감 기준선", "지금": f"MNO {int((plans['carrier_type']=='MNO').sum())}행 · 중앙값 {plans.loc[plans['carrier_type']=='MNO','monthly_fee'].median():,.0f}원",
     "재분류 후": f"대리점 {base_fams}개 그룹 · 중앙값 {fam.loc[fam['channel']=='통신3사 대리점','정상가'].median():,.0f}원"},
    {"항목": "통신3사 행/요금제", "지금": f"{len(mno_side):,}행을 각각 다른 요금제로 취급",
     "재분류 후": f"{len(fam)}개 그룹 + 혜택 선택지"},
])
summary_fix

## 14. 결론 — 코드로 옮길 항목

| # | EDA에서 확인한 것 | 파이프라인 반영 |
|---|---|---|
| 1 | 이름·요금이 같은 프로모션 변형이 다수 | 표시 단계에서 대표 1건 + "유사 N건" 접기 |
| 2 | 할인기간 결측의 뜻이 MVNO/MNO에서 다름 | `discount_months()` 한 함수로 고정, MNO 선택약정=24개월 가정 명시 |
| 3 | `NaN >= N` 이 무제한 요금제를 탈락시킴 | 모든 숫자 조건은 `at_least()` 경유 + 회귀 테스트 |
| 4 | 보이는 요금 순위 ≠ 실제 지출 순위 | 정렬은 24개월 기준 실제 요금으로, 보이는 요금은 화면 표시용 / `promo_cliff` 플래그 |
| 4-B | 페이백형 1건이 원래 요금 자리를 덮어써서 모든 유형의 1위 차지 | 적재 단계 이상치 게이트(교차검증·구간중앙값·`needs_review`) |
| 5 | MNO·MVNO는 가격대가 분리, 브랜드 집중 | 그룹 키에 `carrier_type`, Top-N에 브랜드 다양성 제약 |
| 6 | 데이터는 로그 눈금, 다 쓴 뒤 속도는 5등급 | `log1p(data)` / `sqrt(cost)`, 테더링 `undisclosed`는 미지로 분리 |
| 7 | 1GB당 가격은 구간마다 자릿수가 다름 | 단가로 순위 매기지 않고, 사용량 분포 기반 **기대 청구액**으로 비교 |
| 8 | 3개 항목 비교는 과하게 걸러냄, 6개 항목+그룹이 현실적 | 6개 항목 + (carrier, online, age) 그룹, `dominated_by` 보존 |
| 9 | OTT 환산은 최저가 순위가 아니라 중·고가 순위를 바꿈 | OTT 가치는 개인 구독 여부 기반, 차감 전/후 둘 다 노출 |
| 10 | 가입자 수가 일부 요금제에 쏠려 있고, 가격만으로는 설명이 거의 안 됨 | `log_share` 유지 + **브랜드 고정효과** 필수, 계수=선호 등식 금지 |
| 11 | 카탈로그가 회차마다 수백 건 변동 | 생존일수 기반 `discontinuation_risk`, 백필 구간(~2026-08-10) 제외 |
| 12 | 요금 기준만 바꿔도 Top-5가 흔들림 | 위 결정들을 **회귀 테스트**로 고정, 후보 3개 이하 구간은 조건 완화 |
| 13-A | 너겟·요고·다이렉트 90건은 대리점과 성격이 다름 (모요와 중복 38건) | `channel` 3분류: 후보 = MVNO + 온라인 직판, 기준선 = 대리점. 중복은 필드 단위 병합 |
| 13-B | 통신3사 556행 = 실제 122개 요금제 (혜택 변형) | `plan_family` + `main_benefit`. 통계·후보는 그룹 대표 1건, 화면은 카드 1장 + 혜택 선택지 |

### 다음에 붙일 것

* **사용량 분포 보정**: KISDI/방통위 통계로 LogNormal σ를 맞춘 뒤 기대 청구액 계산 → 7장의 단가 문제 해결
* **결합할인 룩업 테이블**: 지금은 MVNO 쪽 비용 페널티로만 근사 — 실제 표가 들어가면 MNO 기준선이 다시 움직인다
* **σ 추정용 생존 데이터**: 2026-08-11 이후 구간만 누적해서 11장의 위험 점수를 수치화